In [4]:
from google.colab import drive
drive.mount('/content/drive')

!pip install rdkit

import sys
sys.path.append("/content/drive/MyDrive/CompDReAM/src")
from functions import smiles_to_morgan_fp, fetch_uniprot_sequences, batch_embed_sequences

# Core libraries
import pandas as pd
import numpy as np
import json
import time
import requests
import os
import glob
from math import sqrt
from datetime import datetime

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys, rdFMCS
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.DataStructs.cDataStructs import ConvertToNumpyArray
from rdkit.ML.Descriptors import MoleculeDescriptors

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.metrics import (mean_squared_error, r2_score, roc_auc_score, f1_score)
from sklearn.base import clone
import joblib

# PyTorch (ProtBERT)
import torch
from torch.utils.data import DataLoader

# Transformers (ProtBERT)
from transformers import BertTokenizer, BertModel

# Visual
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from IPython.display import display
from tqdm import tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
base_dir = "/content/drive/MyDrive/CompDReAM/v2"
os.makedirs(base_dir, exist_ok=True)

df = pd.read_csv("/content/drive/MyDrive/CompDReAM/training_dataset.csv", low_memory=False)
df = df.dropna(subset=["Canonical SMILES", "UniProt", "pChEMBL"]).reset_index(drop=True)
model_path = "/content/drive/MyDrive/CompDReAM/v2/rf_model.pkl"

generator = GetMorganGenerator(radius=2, fpSize=2048)

fps, valid_idx = [], []
for idx, s in enumerate(df["Canonical SMILES"]):
    arr = smiles_to_morgan_fp(s)
    if arr is not None:
        fps.append(arr)
        valid_idx.append(idx)

df = df.iloc[valid_idx].reset_index(drop=True)
X_mol = np.array(fps)

all_uniprots = df["UniProt"].dropna().str.split(",").explode().str.strip().unique().tolist()
cache_file = "/content/drive/MyDrive/CompDReAM/id_to_seq.json"

if os.path.exists(cache_file):
    print(f"✓ Loading sequences from cache: {cache_file}")
    with open(cache_file) as f:
        id_to_seq = json.load(f)
else:
    print("✗ Cache not found — fetching sequences from UniProt API...")
    id_to_seq = fetch_uniprot_sequences(all_uniprots)
    with open(cache_file, "w") as f:
        json.dump(id_to_seq, f)
    print(f"✓ Sequences fetched and saved to: {cache_file}")

tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").eval()

df["UniProt_List"] = df["UniProt"].str.split(",").apply(lambda lst: [i.strip() for i in lst])
df["BestUniProt"] = df["UniProt_List"].apply(lambda lst: lst[0])  # choose first UniProt ID
df["Sequence"] = df["BestUniProt"].map(id_to_seq)
valid_seq_mask = df["Sequence"].notna()
sequences = df.loc[valid_seq_mask, "Sequence"].tolist()

prot_cache_path = "/content/drive/MyDrive/CompDReAM/v2/X_prot.npy"
batch_prefix = "/content/drive/MyDrive/CompDReAM/protbert/protbert_batch"
batch_files = sorted(glob.glob(f"{batch_prefix}_*.npy"))
if batch_files:
    sample = np.load(batch_files[0], mmap_mode="r")
    dim = sample.shape[1]
    total_rows = sum(np.load(f, mmap_mode="r").shape[0] for f in batch_files)
else:
    dim = None
    total_rows = None
if os.path.exists(prot_cache_path):
    print(f"✓ Loaded final cached embeddings: {prot_cache_path}")
    if dim is not None and total_rows is not None:
        X_prot = np.memmap(prot_cache_path, dtype='float32', mode='r', shape=(total_rows, dim))
    else:
        X_prot = np.load(prot_cache_path)
else:
    print("✗ Final cache not found — checking for batch fragments...")
    if batch_files:
        print(f"✓ Found {len(batch_files)} cached batches. Merging with np.memmap...")
        X_prot_memmap = np.memmap(prot_cache_path, dtype='float32', mode='w+', shape=(total_rows, dim))
        i = 0
        for path in tqdm(batch_files, desc="Merging batches"):
            batch = np.load(path, mmap_mode="r")
            n = batch.shape[0]
            X_prot_memmap[i:i+n, :] = batch
            i += n
        del X_prot_memmap
        print(f"✓ ProtBERT embeddings saved to: {prot_cache_path}")
        X_prot = np.memmap(prot_cache_path, dtype='float32', mode='r', shape=(total_rows, dim))
    else:
        print("✗ No batch fragments found — computing embeddings...")
        X_prot = batch_embed_sequences(sequences, cache_prefix=batch_prefix)
        np.save(prot_cache_path, X_prot)
        print(f"✓ ProtBERT embeddings saved to: {prot_cache_path}")

df = df.loc[valid_seq_mask].reset_index(drop=True)
X_mol = X_mol[valid_seq_mask.values]
y = df["pChEMBL"].astype(float).values

X_combined_path = "/content/drive/MyDrive/CompDReAM/v2/X_combined.npy"
y_path = "/content/drive/MyDrive/CompDReAM/v2/y.npy"
n_samples = X_mol.shape[0]
combined_shape = (n_samples, X_mol.shape[1] + X_prot.shape[1])
if os.path.exists(X_combined_path) and os.path.exists(y_path):
    print(f"✓ Loading existing X and y from disk.")
    X = np.memmap(X_combined_path, dtype=np.float32, mode="r", shape=combined_shape)
    y = np.load(y_path)
else:
    print("✗ Cache not found — creating X_combined.npy and y.npy")
    X_combined = np.memmap(X_combined_path, dtype=np.float32, mode="w+", shape=combined_shape)
    X_combined[:, :X_mol.shape[1]] = X_mol
    X_combined[:, X_mol.shape[1]:] = X_prot
    X_combined.flush()
    np.save(y_path, y)
    print("✓ Features and labels saved.")
    X = np.memmap(X_combined_path, dtype=np.float32, mode="r", shape=combined_shape)

## === Create "pair" column (early in the script!) ===
df["pair"] = df["Canonical SMILES"] + "_" + df["BestUniProt"]

# === Remove duplicates and split by pair ===
unique_pairs = df.drop_duplicates(subset="pair")
train_pairs, test_pairs = train_test_split(unique_pairs["pair"], test_size=0.2, random_state=42)
train_df = df[df["pair"].isin(train_pairs)]
test_df = df[df["pair"].isin(test_pairs)]

assert len(set(train_df["pair"]).intersection(set(test_df["pair"]))) == 0

# === Use correct X and y splits ===
X_train = X[train_df.index]
y_train = y[train_df.index]
X_test = X[test_df.index]
y_test = y[test_df.index]

# === Train or load model ===
if os.path.exists(model_path):
    print("✓ Loading pre-trained model...")
    rf = joblib.load(model_path)
else:
    print("✗ No pre-trained model found. Training from scratch...")
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    joblib.dump(rf, model_path)
    print(f"✓ Model saved to: {model_path}")

# === Evaluate ===
y_pred = rf.predict(X_test)
print("Train R²:", rf.score(X_train, y_train))
print("Test R²:", r2_score(y_test, y_pred))
print("RMSE:", sqrt(mean_squared_error(y_test, y_pred)))

[11:11:33] WARNING: not removing hydrogen atom without neighbors


✓ Loading sequences from cache: /content/drive/MyDrive/CompDReAM/id_to_seq.json
✗ Final cache not found — checking for batch fragments...
✓ Found 927 cached batches. Merging with np.memmap...


Merging batches: 100%|██████████| 927/927 [00:10<00:00, 86.17it/s]


✓ ProtBERT embeddings saved to: /content/drive/MyDrive/CompDReAM/v2/X_prot.npy
✗ Cache not found — creating X_combined.npy and y.npy
✓ Features and labels saved.
✗ No pre-trained model found. Training from scratch...
✓ Model saved to: /content/drive/MyDrive/CompDReAM/v2/rf_model.pkl
Train R²: 0.9386254771427606
Test R²: 0.7348819308878929
RMSE: 0.518145176428569


In [6]:
metadata = {
    "model": "RandomForestRegressor",
    "timestamp": datetime.now().isoformat(),
    "n_estimators": 100,
    "random_state": 42,
    "train_r2": rf.score(X_train, y_train),
    "test_r2": r2_score(y_test, y_pred),
    "rmse": sqrt(mean_squared_error(y_test, y_pred)),
    "n_samples": X.shape[0],
    "feature_dim": X.shape[1],
    "model_path": model_path
}
pd.DataFrame([metadata]).to_csv("/content/drive/MyDrive/CompDReAM/v2/rf_metadata_summary.csv", index=False)
print(f"✓ Metadata saved to rf_metadata_summary.csv")

✓ Metadata saved to rf_metadata_summary.csv
